<a href="https://colab.research.google.com/github/erreesse-rs/trabalho-api/blob/main/tratamento_dados_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import re
import csv
import glob
import zipfile
import unicodedata
import numpy as np
import pandas as pd
from datetime import datetime
from google.colab import files

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

ARQUIVOS_ESPERADOS = [
    'characters.csv',
    'character_creators.csv',
    'character_powers.csv',
    'character_teams.csv',
    'concepts.csv',
    'volume_concepts.csv',
    'volumes.csv'
]

print("Selecione os 7 arquivos CSV para upload:")
uploaded = files.upload()

def mapear_arquivos(arquivos_esperados):
    """Identifica os arquivos mesmo com sufixos de navegador como ' (1).csv'."""
    mapeamento = {}
    arquivos_dir = os.listdir('.')
    for esperado in arquivos_esperados:
        base = esperado.replace('.csv', '')
        padrao = re.compile(rf'^{re.escape(base)}(\s*\(\d+\))?\.csv$', re.IGNORECASE)
        correspondencias = [f for f in arquivos_dir if padrao.match(f)]
        if correspondencias:
            # Prioriza o arquivo mais recente caso haja duplicatas de upload
            correspondencias.sort(key=lambda x: os.path.getmtime(x), reverse=True)
            mapeamento[esperado] = correspondencias[0]
        else:
            print(f"[AVISO] Arquivo não encontrado: {esperado}")
    return mapeamento

mapa_arquivos = mapear_arquivos(ARQUIVOS_ESPERADOS)
print("\nArquivos mapeados com sucesso:")
for k, v in mapa_arquivos.items():
    print(f"  {k:<25} -> {v}")

Selecione os 7 arquivos CSV para upload:


Saving character_creators.csv to character_creators.csv
Saving character_powers.csv to character_powers.csv
Saving characters.csv to characters.csv
Saving character_teams.csv to character_teams.csv
Saving concepts.csv to concepts.csv
Saving volume_concepts.csv to volume_concepts.csv
Saving volumes.csv to volumes.csv

Arquivos mapeados com sucesso:
  characters.csv            -> characters.csv
  character_creators.csv    -> character_creators.csv
  character_powers.csv      -> character_powers.csv
  character_teams.csv       -> character_teams.csv
  concepts.csv              -> concepts.csv
  volume_concepts.csv       -> volume_concepts.csv
  volumes.csv               -> volumes.csv


In [2]:
diagnostico_registros = []
dfs_brutos_str = {}
dfs_originais = {}

def detectar_separador_e_encoding(caminho):
    with open(caminho, 'rb') as f:
        raw = f.read(65536)
    encoding = 'utf-8-sig' if raw.startswith(b'\xef\xbb\xbf') else 'utf-8'
    try:
        amostra = raw.decode(encoding)
    except UnicodeDecodeError:
        encoding = 'latin-1'
        amostra = raw.decode(encoding, errors='replace')
    primeira_linha = amostra.splitlines()[0] if amostra.splitlines() else ''
    sep = ';' if primeira_linha.count(';') >= primeira_linha.count(',') else ','
    return encoding, sep

for nome_logico, caminho in mapa_arquivos.items():
    enc, sep = detectar_separador_e_encoding(caminho)

    # Validação linha a linha via csv.reader (sem descartar nenhuma linha)
    with open(caminho, 'r', encoding=enc, newline='') as f:
        reader = csv.reader(f, delimiter=sep, quotechar='"')
        linhas = list(reader)

    header = linhas[0]
    n_cols = len(header)
    linhas_malformadas = [
        (idx + 1, len(r), r) for idx, r in enumerate(linhas[1:], start=1) if len(r) != n_cols
    ]

    df_str = pd.read_csv(caminho, sep=sep, encoding=enc, dtype=str, keep_default_na=False)
    df_nat = pd.read_csv(caminho, sep=sep, encoding=enc)

    dfs_brutos_str[nome_logico] = df_str
    dfs_originais[nome_logico] = df_nat

    duplicatas = int(df_nat.duplicated().sum())
    vazias = int(df_nat.isna().all(axis=1).sum())

    print(f"\n{'='*75}")
    print(f"ARQUIVO: {nome_logico} | Encoding: {enc} | Separador: '{sep}'")
    print(f"Linhas: {len(df_nat)} | Colunas: {len(df_nat.columns)} | Duplicatas exatas: {duplicatas} | Linhas vazias: {vazias} | Malformadas: {len(linhas_malformadas)}")
    print(f"{'='*75}")

    if linhas_malformadas:
        for num_linha, qtd, conteudo in linhas_malformadas[:5]:
            print(f"  [ERRO ESTRUTURAL] Linha {num_linha}: {qtd} campos (esperado {n_cols}) -> {conteudo}")
            diagnostico_registros.append({
                'arquivo': nome_logico, 'coluna': 'ESTRUTURA', 'tipo_problema': 'Linha malformada',
                'quantidade': 1, 'detalhes': f'Linha {num_linha} com {qtd} campos'
            })

    resumo_cols = []
    for col in df_nat.columns:
        s_raw = df_str[col]
        nulos = int(df_nat[col].isna().sum())
        unicos = int(df_nat[col].nunique(dropna=True))
        espacos = int(s_raw.apply(lambda x: x != x.strip()).sum())
        quebras = int(s_raw.apply(lambda x: '\n' in x or '\r' in x).sum())
        invisiveis = int(s_raw.apply(lambda x: bool(re.search(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\u200b-\u200f\ufeff\u00a0]', x))).sum())
        mojibake = int(s_raw.apply(lambda x: bool(re.search(r'Ã.|Â.|â€|ï¿½|\ufffd', x))).sum())

        resumo_cols.append({
            'coluna': col, 'tipo': str(df_nat[col].dtype), 'nulos': nulos,
            'unicos': unicos, 'espacos_ext': espacos, 'quebras_linha': quebras,
            'invisiveis': invisiveis, 'mojibake': mojibake
        })

        for prob_nome, qtd in [
            ('Espaços nas extremidades', espacos),
            ('Quebras de linha internas', quebras),
            ('Caracteres invisíveis', invisiveis),
            ('Possível mojibake', mojibake)
        ]:
            if qtd > 0:
                diagnostico_registros.append({
                    'arquivo': nome_logico, 'coluna': col, 'tipo_problema': prob_nome,
                    'quantidade': qtd, 'detalhes': f'{qtd} ocorrência(s) detectada(s)'
                })

    print(pd.DataFrame(resumo_cols).to_string(index=False))


ARQUIVO: characters.csv | Encoding: utf-8-sig | Separador: ';'
Linhas: 1000 | Colunas: 15 | Duplicatas exatas: 0 | Linhas vazias: 0 | Malformadas: 0
                    coluna   tipo  nulos  unicos  espacos_ext  quebras_linha  invisiveis  mojibake
                        id  int64      0    1000            0              0           0         0
                      name object      0     996            0              0           0         0
                 real_name object    254     736           13              0           0         0
                   aliases object    296     703            0              0           0         0
                 publisher object      0       2            0              0           0         0
              publisher_id  int64      0       2            0              0           0         0
                    gender  int64      0       3            0              0           0         0
                     birth object    953      45          

In [3]:
pendencias_registros = []

# 1. Auditoria de datas em characters.csv
ch = dfs_originais['characters.csv']
for dcol in ['birth', 'date_added', 'date_last_updated']:
    nao_nulos = ch[dcol].dropna()
    convertidos = pd.to_datetime(nao_nulos, errors='coerce')
    falhas = nao_nulos[convertidos.isna()]
    print(f"Data '{dcol}' em characters.csv -> Não nulos: {len(nao_nulos)} | Falhas de parse padrão: {len(falhas)}")
    for idx, val in falhas.items():
        print(f"  [PENDÊNCIA DATA] Índice {idx} (id={ch.loc[idx, 'id']}, name={ch.loc[idx, 'name']}): '{val}'")
        diagnostico_registros.append({
            'arquivo': 'characters.csv', 'coluna': dcol, 'tipo_problema': 'Data fora do limite Timestamp',
            'quantidade': 1, 'detalhes': f"Registro id={ch.loc[idx, 'id']} com valor '{val}'"
        })
        pendencias_registros.append({
            'arquivo': 'characters.csv', 'coluna': dcol, 'registro_id': ch.loc[idx, 'id'],
            'valor_original': val, 'motivo': 'Data fictícia/futurista fora do limite de timestamp (ano > 2262). Mantida em texto ISO.'
        })

# 2. Auditoria de categorias e domínios esperados
print("\nEditoras em characters.csv:")
print(ch.groupby(['publisher', 'publisher_id']).size())

print("\nEditoras em volumes.csv:")
print(dfs_originais['volumes.csv'].groupby(['publisher', 'publisher_id']).size())

print("\nGênero (gender) em characters.csv (0=Outro/Desconhecido, 1=Masculino, 2=Feminino):")
print(ch['gender'].value_counts(dropna=False))

print("\nOrigem (origin) em characters.csv:")
print(ch['origin'].value_counts(dropna=False))

# 3. Verificação de deslocamento de colunas (ex: 'Jr.' em publisher ou colunas numéricas com texto)
for nome, df in dfs_originais.items():
    if 'publisher' in df.columns:
        estranhos = df[~df['publisher'].isin(['Marvel', 'DC Comics'])]
        if len(estranhos) > 0:
            print(f"[ALERTA DESLOCAMENTO] {nome}.publisher possui {len(estranhos)} valores inesperados!")
    for col in df.columns:
        if col.endswith('_id') or col in ['id', 'start_year', 'count_of_issues', 'count_of_issue_appearances']:
            nao_num = pd.to_numeric(df[col], errors='coerce').isna() & df[col].notna()
            if nao_num.sum() > 0:
                print(f"[ALERTA TIPO] {nome}.{col} possui {nao_num.sum()} valores não numéricos!")

Data 'birth' em characters.csv -> Não nulos: 47 | Falhas de parse padrão: 1
  [PENDÊNCIA DATA] Índice 554 (id=1786, name=Booster Gold): 'Dec 29, 2442'
Data 'date_added' em characters.csv -> Não nulos: 1000 | Falhas de parse padrão: 0
Data 'date_last_updated' em characters.csv -> Não nulos: 1000 | Falhas de parse padrão: 0

Editoras em characters.csv:
publisher  publisher_id
DC Comics  10              500
Marvel     31              500
dtype: int64

Editoras em volumes.csv:
publisher  publisher_id
DC Comics  10              500
Marvel     31              500
dtype: int64

Gênero (gender) em characters.csv (0=Outro/Desconhecido, 1=Masculino, 2=Feminino):
gender
1    640
2    285
0     75
Name: count, dtype: int64

Origem (origin) em characters.csv:
origin
Human          564
Mutant         120
Alien           63
Other           60
NaN             59
God/Eternal     57
Radiation       24
Cyborg          21
Robot           19
Animal          13
Name: count, dtype: int64


In [4]:
def verificar_integridade(dfs_dict):
    relacoes = [
        ('character_creators.csv', 'character_id', 'characters.csv', 'id', 'N:1 (M:N com criadores)'),
        ('character_powers.csv',   'character_id', 'characters.csv', 'id', 'N:1 (M:N com poderes)'),
        ('character_teams.csv',    'character_id', 'characters.csv', 'id', 'N:1 (M:N com equipes)'),
        ('volume_concepts.csv',    'volume_id',    'volumes.csv',    'volume_id',  'N:1 (M:N com conceitos)'),
        ('volume_concepts.csv',    'concept_id',   'concepts.csv',   'concept_id', 'N:1 (M:N com volumes)'),
    ]
    resultados = []
    for f_filha, col_fk, f_pai, col_pk, card in relacoes:
        df_f = dfs_dict[f_filha]
        df_p = dfs_dict[f_pai]
        pks = set(df_p[col_pk].dropna())
        corresp = int(df_f[col_fk].isin(pks).sum())
        orfas = int((~df_f[col_fk].isin(pks)).sum())
        ids_repetidos_filha = int(len(df_f) - df_f[col_fk].nunique())
        resultados.append({
            'tabela_origem_fk': f_filha, 'coluna_fk': col_fk,
            'tabela_destino_pk': f_pai,  'coluna_pk': col_pk,
            'total_registros': len(df_f), 'correspondencias': corresp,
            'referencias_orfas': orfas, 'ids_repetidos_fk': ids_repetidos_filha,
            'cardinalidade': card, 'status': 'ÍNTEGRA' if orfas == 0 else 'COM ÓRFÃS'
        })
    return pd.DataFrame(resultados)

df_integridade_antes = verificar_integridade(dfs_originais)
print("=== INTEGRIDADE REFERENCIAL (ANTES DO TRATAMENTO) ===")
print(df_integridade_antes.to_string(index=False))

# Verificar duplicidade de chaves compostas nas tabelas associativas
chaves_compostas = [
    ('character_creators.csv', ['character_id', 'creator_id']),
    ('character_powers.csv',   ['character_id', 'power_id']),
    ('character_teams.csv',    ['character_id', 'team_id']),
    ('volume_concepts.csv',    ['volume_id', 'concept_id']),
]
print("\n=== CHAVES COMPOSTAS DUPLICADAS ===")
for f, cols in chaves_compostas:
    dups = int(dfs_originais[f].duplicated(subset=cols).sum())
    print(f"  {f} {cols}: {dups} duplicata(s)")

# Tabela de Relacionamentos e Complementaridade (Etapa 4)
df_relacionamentos = pd.DataFrame([
    {'tabela': 'characters.csv',         'papel': 'Cadastral (Dimensão)',  'chave_primaria': 'id',                          'chaves_estrangeiras': 'publisher_id',           'dominio': 'Personagens'},
    {'tabela': 'character_creators.csv', 'papel': 'Associativa (M:N)',     'chave_primaria': '(character_id, creator_id)',  'chaves_estrangeiras': 'character_id',           'dominio': 'Personagens'},
    {'tabela': 'character_powers.csv',   'papel': 'Associativa (M:N)',     'chave_primaria': '(character_id, power_id)',    'chaves_estrangeiras': 'character_id',           'dominio': 'Personagens'},
    {'tabela': 'character_teams.csv',    'papel': 'Associativa (M:N)',     'chave_primaria': '(character_id, team_id)',     'chaves_estrangeiras': 'character_id',           'dominio': 'Personagens'},
    {'tabela': 'volumes.csv',            'papel': 'Cadastral (Dimensão)',  'chave_primaria': 'volume_id',                   'chaves_estrangeiras': 'publisher_id',           'dominio': 'Volumes/Conceitos'},
    {'tabela': 'concepts.csv',           'papel': 'Cadastral (Dimensão)',  'chave_primaria': 'concept_id',                  'chaves_estrangeiras': 'Nenhuma',                'dominio': 'Volumes/Conceitos'},
    {'tabela': 'volume_concepts.csv',    'papel': 'Associativa (M:N)',     'chave_primaria': '(volume_id, concept_id)',     'chaves_estrangeiras': 'volume_id, concept_id',  'dominio': 'Volumes/Conceitos'},
])
print("\n=== ESTRUTURA E COMPLEMENTARIDADE ===")
print(df_relacionamentos.to_string(index=False))

=== INTEGRIDADE REFERENCIAL (ANTES DO TRATAMENTO) ===
      tabela_origem_fk    coluna_fk tabela_destino_pk  coluna_pk  total_registros  correspondencias  referencias_orfas  ids_repetidos_fk           cardinalidade  status
character_creators.csv character_id    characters.csv         id             1417              1417                  0               761 N:1 (M:N com criadores) ÍNTEGRA
  character_powers.csv character_id    characters.csv         id             9014              9014                  0              8213   N:1 (M:N com poderes) ÍNTEGRA
   character_teams.csv character_id    characters.csv         id             5437              5437                  0              4684   N:1 (M:N com equipes) ÍNTEGRA
   volume_concepts.csv    volume_id       volumes.csv  volume_id             1157              1157                  0               712 N:1 (M:N com conceitos) ÍNTEGRA
   volume_concepts.csv   concept_id      concepts.csv concept_id             1157              1157  

In [5]:
correcoes_log = []
dfs_tratados = {}

def padronizar_data_segura(valor):
    """Converte datas como 'Apr 23, 1956' ou 'Dec 29, 2442' para ISO 'YYYY-MM-DD' sem gerar NaT."""
    if pd.isna(valor) or str(valor).strip() == '':
        return np.nan
    s = str(valor).strip()
    for fmt in ('%b %d, %Y', '%Y-%m-%d %H:%M:%S', '%Y-%m-%d'):
        try:
            dt = datetime.strptime(s, fmt)
            return dt.strftime('%Y-%m-%d' if fmt == '%b %d, %Y' else '%Y-%m-%d %H:%M:%S')
        except ValueError:
            continue
    return s

for nome_logico, df_orig in dfs_originais.items():
    df = df_orig.copy()

    # 1. Padronização de nomes de colunas (snake_case, sem espaços)
    novas_cols = {c: c.strip().lower().replace(' ', '_') for c in df.columns}
    df.rename(columns=novas_cols, inplace=True)

    # Identificador para log de correções
    id_col = df.columns[0]

    # 2. Normalização Unicode (NFC) e remoção de espaços nas extremidades em colunas texto
    for col in df.select_dtypes(include='object').columns:
        for idx, val in df[col].items():
            if pd.isna(val):
                continue
            novo_val = unicodedata.normalize('NFC', str(val)).strip()
            # Remover caracteres invisíveis caso existam
            novo_val = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\u200b-\u200f\ufeff]', '', novo_val)
            if novo_val != val:
                correcoes_log.append({
                    'arquivo': nome_logico, 'coluna': col,
                    'registro_id': df.loc[idx, id_col],
                    'valor_anterior': repr(val), 'valor_posterior': repr(novo_val),
                    'acao': 'Limpeza de espaços / normalização Unicode'
                })
                df.at[idx, col] = novo_val if novo_val != '' else np.nan

    # 3. Padronização de datas em characters.csv
    if nome_logico == 'characters.csv':
        for dcol in ['birth', 'date_added', 'date_last_updated']:
            for idx, val in df[dcol].items():
                if pd.isna(val):
                    continue
                novo_val = padronizar_data_segura(val)
                if novo_val != val:
                    correcoes_log.append({
                        'arquivo': nome_logico, 'coluna': dcol,
                        'registro_id': df.loc[idx, id_col],
                        'valor_anterior': val, 'valor_posterior': novo_val,
                        'acao': 'Padronização de formato de data ISO'
                    })
                    df.at[idx, dcol] = novo_val

    # 4. Garantia de tipos inteiros nas colunas de ID e contagens
    for col in df.columns:
        if col.endswith('_id') or col in ['id', 'gender', 'start_year', 'count_of_issues', 'count_of_issue_appearances']:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    dfs_tratados[nome_logico] = df

print(f"Tratamento concluído! Total de correções individuais registradas: {len(correcoes_log)}")

Tratamento concluído! Total de correções individuais registradas: 98


In [10]:
pasta_saida = 'dados_tratados'
os.makedirs(pasta_saida, exist_ok=True)

ch_t  = dfs_tratados['characters.csv'].copy()
cc_t  = dfs_tratados['character_creators.csv']
cp_t  = dfs_tratados['character_powers.csv']
ct_t  = dfs_tratados['character_teams.csv']
vc_t  = dfs_tratados['volume_concepts.csv']

# 1. Agregar criadores, poderes e equipes no nível do personagem (1 linha = 1 personagem)
agg_creators = cc_t.groupby('character_id').agg(
    qtd_criadores=('creator_id', 'nunique'),
    criadores=('creator_name', lambda s: ' | '.join(sorted(s.dropna().unique())))
)

agg_powers = cp_t.groupby('character_id').agg(
    qtd_poderes=('power_id', 'nunique'),
    poderes=('power_name', lambda s: ' | '.join(sorted(s.dropna().unique())))
)

agg_teams = ct_t.groupby('character_id').agg(
    qtd_equipes=('team_id', 'nunique'),
    equipes=('team_name', lambda s: ' | '.join(sorted(s.dropna().unique())))
)

# 2. Montar a base consolidada de personagens
base_personagens_consolidada = (
    ch_t.merge(agg_creators, left_on='id', right_index=True, how='left')
        .merge(agg_powers,   left_on='id', right_index=True, how='left')
        .merge(agg_teams,    left_on='id', right_index=True, how='left')
)

for col_qtd in ['qtd_criadores', 'qtd_poderes', 'qtd_equipes']:
    base_personagens_consolidada[col_qtd] = base_personagens_consolidada[col_qtd].fillna(0).astype(int)

# Tradução amigável de gênero para filtros do dashboard
mapa_genero = {1: 'Masculino', 2: 'Feminino', 0: 'Outro/Desconhecido'}
base_personagens_consolidada['gender_label'] = base_personagens_consolidada['gender'].map(mapa_genero)

# 3. Tabela auxiliar de conceitos por editora (domínio Volumes/Conceitos)
resumo_conceitos_editora = (
    vc_t.groupby(['publisher', 'publisher_id', 'concept_id', 'concept_name'], as_index=False)
        .agg(qtd_volumes=('volume_id', 'nunique'))
        .sort_values(['publisher', 'qtd_volumes'], ascending=[True, False])
)

print(f"Base consolidada gerada: {base_personagens_consolidada.shape[0]} linhas x {base_personagens_consolidada.shape[1]} colunas")
print(base_personagens_consolidada[['id', 'name', 'publisher', 'count_of_issue_appearances', 'qtd_criadores', 'qtd_poderes', 'qtd_equipes', 'criadores']].sort_values('count_of_issue_appearances', ascending=False).head(5).to_string(index=False))

# 4. Salvar os 7 arquivos tratados originais + 2 bases prontas para dashboard + 5 relatórios
for nome, df in dfs_tratados.items():
    df.to_csv(os.path.join(pasta_saida, nome.replace('.csv', '_tratado.csv')), sep=';', encoding='utf-8-sig', index=False, quoting=csv.QUOTE_MINIMAL)

base_personagens_consolidada.to_csv(os.path.join(pasta_saida, 'base_personagens_consolidada.csv'), sep=';', encoding='utf-8-sig', index=False, quoting=csv.QUOTE_MINIMAL)
resumo_conceitos_editora.to_csv(os.path.join(pasta_saida, 'resumo_conceitos_editora.csv'), sep=';', encoding='utf-8-sig', index=False, quoting=csv.QUOTE_MINIMAL)

pd.DataFrame(diagnostico_registros).to_csv(os.path.join(pasta_saida, 'relatorio_diagnostico.csv'), sep=';', encoding='utf-8-sig', index=False)
pd.DataFrame(correcoes_log).to_csv(os.path.join(pasta_saida, 'relatorio_correcoes.csv'), sep=';', encoding='utf-8-sig', index=False)
df_integridade_depois.to_csv(os.path.join(pasta_saida, 'relatorio_integridade.csv'), sep=';', encoding='utf-8-sig', index=False)
df_relacionamentos.to_csv(os.path.join(pasta_saida, 'relatorio_relacionamentos.csv'), sep=';', encoding='utf-8-sig', index=False)
pd.DataFrame(pendencias_registros).to_csv(os.path.join(pasta_saida, 'relatorio_pendencias.csv'), sep=';', encoding='utf-8-sig', index=False)

# 5. Exportar também em arquivo único Excel (.xlsx) pronto para Power BI / Looker / Excel
!pip install -q openpyxl
excel_path = os.path.join(pasta_saida, 'base_dashboard_marvel_dc.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    base_personagens_consolidada.to_excel(writer, sheet_name='Personagens_Consolidado', index=False)
    resumo_conceitos_editora.to_excel(writer, sheet_name='Conceitos_por_Editora', index=False)
    cc_t.to_excel(writer, sheet_name='Rel_Criadores', index=False)
    cp_t.to_excel(writer, sheet_name='Rel_Poderes', index=False)
    ct_t.to_excel(writer, sheet_name='Rel_Equipes', index=False)
    dfs_tratados['volumes.csv'].to_excel(writer, sheet_name='Volumes', index=False)

# 6. Compactar tudo em um único ZIP para download
zip_path = 'dados_tratados_marvel_dc.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for arq in sorted(os.listdir(pasta_saida)):
        zf.write(os.path.join(pasta_saida, arq), arcname=arq)

print(f"\nZIP final pronto: {zip_path} ({len(os.listdir(pasta_saida))} arquivos)")
files.download(zip_path)

Base consolidada gerada: 1000 linhas x 22 colunas
  id            name publisher  count_of_issue_appearances  qtd_criadores  qtd_poderes  qtd_equipes                                         criadores
1699          Batman DC Comics                       25241              2           15           52                            Bill Finger | Bob Kane
1807        Superman DC Comics                       19945              2           34           45                        Jerry Siegel | Joe Shuster
1443      Spider-Man    Marvel                       18160              2           13           53                            Stan Lee | Steve Ditko
1440       Wolverine    Marvel                       16966              4           24           59 Herb Trimpe | John Romita | Len Wein | Roy Thomas
1442 Captain America    Marvel                       12310              2           18           31                            Jack Kirby | Joe Simon

ZIP final pronto: dados_tratados_marvel_dc.zip (1

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>